# AIO Strategy Backtest: MACD MTF + Trendline Breaks + iFVG + ATR SL

Replica en Python la lógica del indicador/estrategia AIO de TradingView:
1. **MACD Multi-Timeframe** — Cruce de MACD/Señal
2. **Trendlines with Breaks** — Pivots + slope dinámico ATR → ruptura
3. **iFVG (Fair Value Gap)** — Detección de imbalances para entrada
4. **ATR Stop Loss Finder** — SL en `high + ATR*mult` / `low - ATR*mult`
5. **Confluencia** — TL break + MACD cross en ventana → Señal
6. **Backtest vectorizado** con métricas y gráficos

In [ ]:
# ── Instalar dependencias si es necesario ──
# !pip install pandas numpy matplotlib ta MetaTrader5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
print('Librerías cargadas ✓')

## 1. Cargar datos
Puedes usar un CSV local o descargar directamente de MetaTrader 5.

In [ ]:
# ═══════════════════════════════════════════════════════════
# OPCIÓN A: Cargar desde CSV local
# ═══════════════════════════════════════════════════════════
CSV_PATH = r'../notebook/datos_locales.csv'   # Ajusta la ruta

try:
    df = pd.read_csv(CSV_PATH, parse_dates=['time'])
    df.set_index('time', inplace=True)
    print(f'CSV cargado: {len(df)} filas')
except FileNotFoundError:
    print('CSV no encontrado. Usando OPCIÓN B (MetaTrader5)...')
    df = None

# ═══════════════════════════════════════════════════════════
# OPCIÓN B: Descargar de MetaTrader 5
# ═══════════════════════════════════════════════════════════
if df is None:
    try:
        import MetaTrader5 as mt5
        if not mt5.initialize():
            raise RuntimeError(f'MT5 init failed: {mt5.last_error()}')
        
        SYMBOL = 'EURUSD'
        TF     = mt5.TIMEFRAME_M5
        BARS   = 10000
        
        rates = mt5.copy_rates_from_pos(SYMBOL, TF, 0, BARS)
        mt5.shutdown()
        
        df = pd.DataFrame(rates)
        df['time'] = pd.to_datetime(df['time'], unit='s')
        df.set_index('time', inplace=True)
        df.rename(columns={'tick_volume': 'volume'}, inplace=True)
        print(f'MT5 descargado: {len(df)} velas de {SYMBOL}')
    except Exception as e:
        print(f'Error MT5: {e}')
        # Generar datos sintéticos para demo
        np.random.seed(42)
        n = 5000
        dates = pd.date_range('2025-01-01', periods=n, freq='5min')
        price = 1.08 + np.cumsum(np.random.randn(n) * 0.0003)
        df = pd.DataFrame({
            'open':  price,
            'high':  price + np.abs(np.random.randn(n)) * 0.0005,
            'low':   price - np.abs(np.random.randn(n)) * 0.0005,
            'close': price + np.random.randn(n) * 0.0002,
            'volume': np.random.randint(100, 5000, n)
        }, index=dates)
        df['high'] = df[['open','close','high']].max(axis=1)
        df['low']  = df[['open','close','low']].min(axis=1)
        print(f'Datos sintéticos generados: {len(df)} velas')

df.head()

## 2. Parámetros de la estrategia

In [ ]:
# ═══ MACD ═══
FAST_LEN   = 12
SLOW_LEN   = 26
SIGNAL_LEN = 9

# ═══ Trendlines ═══
TL_LENGTH  = 14
TL_MULT    = 1.0

# ═══ FVG ═══
FVG_GAP_PCT = 0.009  # % mínimo de gap

# ═══ ATR Stop Loss ═══
ATR_SL_LEN  = 14
ATR_SL_MULT = 1.5

# ═══ Señales / Riesgo ═══
ENTRY_WINDOW = 3     # velas de confirmación
RR_RATIO     = 3.0   # ratio R:R
BE_MULT      = 2.0   # multiplicador Break Even
USE_BE       = True
INITIAL_CAP  = 10000
RISK_PCT     = 2.0   # % de capital por trade

print('Parámetros configurados ✓')

## 3. Calcular indicadores

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3.1 — MACD
# ═══════════════════════════════════════════════════════════
df['ema_fast'] = df['close'].ewm(span=FAST_LEN, adjust=False).mean()
df['ema_slow'] = df['close'].ewm(span=SLOW_LEN, adjust=False).mean()
df['macd']     = df['ema_fast'] - df['ema_slow']
df['macd_sig'] = df['macd'].rolling(SIGNAL_LEN).mean()
df['macd_hist'] = df['macd'] - df['macd_sig']

df['macd_above'] = df['macd'] >= df['macd_sig']
df['macd_cross'] = df['macd_above'] != df['macd_above'].shift(1)
df['cross_up']   = df['macd_cross'] & df['macd_above']
df['cross_down'] = df['macd_cross'] & ~df['macd_above']

print(f'MACD: {df["cross_up"].sum()} cruces alcistas, {df["cross_down"].sum()} cruces bajistas')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3.2 — Trendlines with Breaks (pivot + slope ATR)
# ═══════════════════════════════════════════════════════════
def detect_pivots(df, lookback):
    """Detecta pivot highs y pivot lows con confirmación de lookback velas."""
    ph = pd.Series(np.nan, index=df.index)
    pl = pd.Series(np.nan, index=df.index)
    
    for i in range(lookback, len(df) - lookback):
        window_high = df['high'].iloc[i-lookback:i+lookback+1]
        window_low  = df['low'].iloc[i-lookback:i+lookback+1]
        
        if df['high'].iloc[i] == window_high.max():
            ph.iloc[i] = df['high'].iloc[i]
        if df['low'].iloc[i] == window_low.min():
            pl.iloc[i] = df['low'].iloc[i]
    
    return ph, pl

# ATR para slope
df['tr'] = np.maximum(
    df['high'] - df['low'],
    np.maximum(
        np.abs(df['high'] - df['close'].shift(1)),
        np.abs(df['low'] - df['close'].shift(1))
    )
)
df['atr_tl'] = df['tr'].rolling(TL_LENGTH).mean()
df['tl_slope'] = df['atr_tl'] / TL_LENGTH * TL_MULT

# Pivots
df['pivot_high'], df['pivot_low'] = detect_pivots(df, TL_LENGTH)

# Construir trendlines dinámicas
tl_upper = np.zeros(len(df))
tl_lower = np.zeros(len(df))
slope_ph = np.zeros(len(df))
slope_pl = np.zeros(len(df))
tl_upos  = np.zeros(len(df), dtype=int)
tl_dnos  = np.zeros(len(df), dtype=int)

for i in range(1, len(df)):
    s = df['tl_slope'].iloc[i] if not np.isnan(df['tl_slope'].iloc[i]) else 0
    
    if not np.isnan(df['pivot_high'].iloc[i]):
        slope_ph[i] = s
        tl_upper[i] = df['pivot_high'].iloc[i]
        tl_upos[i]  = 0
    else:
        slope_ph[i] = slope_ph[i-1]
        tl_upper[i] = tl_upper[i-1] - slope_ph[i]
        # check break
        ref = tl_upper[i] - slope_ph[i] * TL_LENGTH
        if df['close'].iloc[i] > ref:
            tl_upos[i] = 1
        else:
            tl_upos[i] = tl_upos[i-1]
    
    if not np.isnan(df['pivot_low'].iloc[i]):
        slope_pl[i] = s
        tl_lower[i] = df['pivot_low'].iloc[i]
        tl_dnos[i]  = 0
    else:
        slope_pl[i] = slope_pl[i-1]
        tl_lower[i] = tl_lower[i-1] + slope_pl[i]
        ref = tl_lower[i] + slope_pl[i] * TL_LENGTH
        if df['close'].iloc[i] < ref:
            tl_dnos[i] = 1
        else:
            tl_dnos[i] = tl_dnos[i-1]

df['tl_upper'] = tl_upper
df['tl_lower'] = tl_lower
df['tl_upos']  = tl_upos
df['tl_dnos']  = tl_dnos

# Rupturas
df['tl_break_up'] = (df['tl_upos'] > df['tl_upos'].shift(1)).fillna(False)
df['tl_break_dn'] = (df['tl_dnos'] > df['tl_dnos'].shift(1)).fillna(False)

print(f'Trendlines: {df["tl_break_up"].sum()} rupturas alcistas, {df["tl_break_dn"].sum()} rupturas bajistas')
print(f'Pivots: {df["pivot_high"].notna().sum()} highs, {df["pivot_low"].notna().sum()} lows')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3.3 — iFVG (Fair Value Gap) Detection
# ═══════════════════════════════════════════════════════════
def detect_fvg(df, gap_pct=0.009):
    """Detecta FVGs alcistas y bajistas."""
    n = len(df)
    fvg_type = np.zeros(n)  # 1=bull, -1=bear
    fvg_top  = np.full(n, np.nan)
    fvg_btm  = np.full(n, np.nan)
    
    h = df['high'].values
    l = df['low'].values
    c = df['close'].values
    
    for i in range(2, n):
        # Bullish: low[i] > high[i-2] and close[i-1] > high[i-2]
        if l[i] > h[i-2] and c[i-1] > h[i-2] and (l[i] - h[i-2]) / h[i-2] * 100 > gap_pct:
            fvg_type[i] = 1
            fvg_top[i]  = l[i]      # top del gap
            fvg_btm[i]  = h[i-2]    # bottom del gap
        # Bearish: high[i] < low[i-2] and close[i-1] < low[i-2]
        elif h[i] < l[i-2] and c[i-1] < l[i-2] and (l[i-2] - h[i]) / h[i] * 100 > gap_pct:
            fvg_type[i] = -1
            fvg_top[i]  = l[i-2]    # top del gap
            fvg_btm[i]  = h[i]      # bottom del gap
    
    return fvg_type, fvg_top, fvg_btm

df['fvg_type'], df['fvg_top'], df['fvg_btm'] = detect_fvg(df, FVG_GAP_PCT)

n_bull_fvg = (df['fvg_type'] == 1).sum()
n_bear_fvg = (df['fvg_type'] == -1).sum()
print(f'FVGs: {n_bull_fvg} alcistas, {n_bear_fvg} bajistas')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3.4 — ATR Stop Loss Finder
# ═══════════════════════════════════════════════════════════
# Usando RMA (Wilder's smoothing) = EWM con alpha=1/length
df['atr_sl'] = df['tr'].ewm(alpha=1/ATR_SL_LEN, adjust=False).mean() * ATR_SL_MULT
df['atr_short_sl'] = df['high'] + df['atr_sl']   # SL para SHORT
df['atr_long_sl']  = df['low']  - df['atr_sl']   # SL para LONG

print(f'ATR SL: último Short SL = {df["atr_short_sl"].iloc[-1]:.5f}, Long SL = {df["atr_long_sl"].iloc[-1]:.5f}')

## 4. Generar señales de confluencia

In [ ]:
# ═══════════════════════════════════════════════════════════
# Contadores de velas desde último evento + señales
# ═══════════════════════════════════════════════════════════
n = len(df)
bars_tl_dn = np.full(n, 999)
bars_tl_up = np.full(n, 999)
bars_mc_dn = np.full(n, 999)
bars_mc_up = np.full(n, 999)

tl_break_up = df['tl_break_up'].values
tl_break_dn = df['tl_break_dn'].values
cross_up    = df['cross_up'].values
cross_dn    = df['cross_down'].values

for i in range(1, n):
    bars_tl_dn[i] = 0 if tl_break_dn[i] else min(bars_tl_dn[i-1] + 1, 999)
    bars_tl_up[i] = 0 if tl_break_up[i] else min(bars_tl_up[i-1] + 1, 999)
    bars_mc_dn[i] = 0 if cross_dn[i]     else min(bars_mc_dn[i-1] + 1, 999)
    bars_mc_up[i] = 0 if cross_up[i]      else min(bars_mc_up[i-1] + 1, 999)

# Señales de confluencia
sell_sig = ((tl_break_dn & (bars_mc_dn <= ENTRY_WINDOW)) |
            (cross_dn & (bars_tl_dn <= ENTRY_WINDOW) & (bars_tl_dn > 0)))
buy_sig  = ((tl_break_up & (bars_mc_up <= ENTRY_WINDOW)) |
            (cross_up & (bars_tl_up <= ENTRY_WINDOW) & (bars_tl_up > 0)))

df['sell_signal'] = sell_sig
df['buy_signal']  = buy_sig

# Rastrear último FVG para precio de entrada
last_bull_top = np.full(n, np.nan)
last_bull_btm = np.full(n, np.nan)
last_bear_top = np.full(n, np.nan)
last_bear_btm = np.full(n, np.nan)

fvg_t = df['fvg_type'].values
fvg_top_v = df['fvg_top'].values
fvg_btm_v = df['fvg_btm'].values

for i in range(1, n):
    last_bull_top[i] = fvg_top_v[i] if fvg_t[i] == 1 else last_bull_top[i-1]
    last_bull_btm[i] = fvg_btm_v[i] if fvg_t[i] == 1 else last_bull_btm[i-1]
    last_bear_top[i] = fvg_top_v[i] if fvg_t[i] == -1 else last_bear_top[i-1]
    last_bear_btm[i] = fvg_btm_v[i] if fvg_t[i] == -1 else last_bear_btm[i-1]

df['last_bull_top'] = last_bull_top
df['last_bull_btm'] = last_bull_btm
df['last_bear_top'] = last_bear_top
df['last_bear_btm'] = last_bear_btm

print(f'Señales: {df["buy_signal"].sum()} BUY, {df["sell_signal"].sum()} SELL')

## 5. Backtest event-driven

In [ ]:
# ═══════════════════════════════════════════════════════════
# Backtest con gestión de posiciones, SL, TP, Break Even
# ═══════════════════════════════════════════════════════════
class Position:
    def __init__(self, direction, entry_px, sl, tp, be_target, lots, bar_idx):
        self.direction = direction   # 'long' or 'short'
        self.entry_px  = entry_px
        self.sl        = sl
        self.tp        = tp
        self.be_target = be_target
        self.lots      = lots
        self.bar_idx   = bar_idx
        self.be_active = False
        self.exit_px   = None
        self.exit_bar  = None
        self.pnl       = 0

def run_backtest(df, initial_capital=10000, risk_pct=2.0):
    capital  = initial_capital
    equity   = [capital]
    trades   = []
    position = None
    
    h = df['high'].values
    l = df['low'].values
    o = df['open'].values
    c = df['close'].values
    buy_s  = df['buy_signal'].values
    sell_s = df['sell_signal'].values
    atr_short_sl = df['atr_short_sl'].values
    atr_long_sl  = df['atr_long_sl'].values
    lb_top = df['last_bull_top'].values
    lb_btm = df['last_bear_btm'].values
    
    for i in range(1, len(df)):
        # ── Gestionar posición abierta ──
        if position is not None:
            exit_px = None
            
            if position.direction == 'long':
                # Check SL
                if l[i] <= position.sl:
                    exit_px = position.sl
                # Check TP
                elif h[i] >= position.tp:
                    exit_px = position.tp
                # Check BE
                elif USE_BE and not position.be_active and h[i] >= position.be_target:
                    position.be_active = True
                    position.sl = position.entry_px  # mover SL a entry
            
            elif position.direction == 'short':
                if h[i] >= position.sl:
                    exit_px = position.sl
                elif l[i] <= position.tp:
                    exit_px = position.tp
                elif USE_BE and not position.be_active and l[i] <= position.be_target:
                    position.be_active = True
                    position.sl = position.entry_px
            
            if exit_px is not None:
                if position.direction == 'long':
                    position.pnl = (exit_px - position.entry_px) * position.lots * 100000
                else:
                    position.pnl = (position.entry_px - exit_px) * position.lots * 100000
                position.exit_px  = exit_px
                position.exit_bar = i
                capital += position.pnl
                trades.append(position)
                position = None
        
        # ── Abrir nueva posición ──
        if position is None:
            if buy_s[i] and not np.isnan(lb_top[i]):
                entry = lb_top[i]   # entrada en FVG top (para buy)
                sl    = atr_long_sl[i]
                risk  = abs(entry - sl)
                if risk > 0:
                    tp = entry + risk * RR_RATIO
                    be = entry + risk * BE_MULT
                    risk_money = capital * risk_pct / 100
                    lots = risk_money / (risk * 100000) if risk * 100000 > 0 else 0.01
                    lots = max(0.01, round(lots, 2))
                    position = Position('long', entry, sl, tp, be, lots, i)
            
            elif sell_s[i] and not np.isnan(lb_btm[i]):
                entry = lb_btm[i]   # entrada en FVG btm (para sell)
                sl    = atr_short_sl[i]
                risk  = abs(sl - entry)
                if risk > 0:
                    tp = entry - risk * RR_RATIO
                    be = entry - risk * BE_MULT
                    risk_money = capital * risk_pct / 100
                    lots = risk_money / (risk * 100000) if risk * 100000 > 0 else 0.01
                    lots = max(0.01, round(lots, 2))
                    position = Position('short', entry, sl, tp, be, lots, i)
        
        equity.append(capital + (0 if position is None else 0))  # mark-to-market simplificado
    
    return trades, equity

trades, equity = run_backtest(df, INITIAL_CAP, RISK_PCT)
print(f'Backtest completado: {len(trades)} trades ejecutados')

## 6. Métricas de rendimiento

In [ ]:
# ═══════════════════════════════════════════════════════════
# Métricas del Backtest
# ═══════════════════════════════════════════════════════════
if len(trades) > 0:
    pnls = np.array([t.pnl for t in trades])
    wins = pnls[pnls > 0]
    losses = pnls[pnls < 0]
    
    total_pnl    = pnls.sum()
    win_rate     = len(wins) / len(pnls) * 100 if len(pnls) > 0 else 0
    avg_win      = wins.mean() if len(wins) > 0 else 0
    avg_loss     = losses.mean() if len(losses) > 0 else 0
    profit_factor = abs(wins.sum() / losses.sum()) if len(losses) > 0 and losses.sum() != 0 else float('inf')
    max_dd       = 0
    peak         = equity[0]
    for e in equity:
        peak = max(peak, e)
        dd   = (peak - e) / peak * 100
        max_dd = max(max_dd, dd)
    
    # Trades por dirección
    longs  = [t for t in trades if t.direction == 'long']
    shorts = [t for t in trades if t.direction == 'short']
    be_trades = [t for t in trades if t.be_active]
    
    metrics = pd.DataFrame({
        'Métrica': [
            'Total Trades', 'Longs', 'Shorts',
            'Win Rate (%)', 'Profit Factor',
            'P&L Total ($)', 'Avg Win ($)', 'Avg Loss ($)',
            'Max Drawdown (%)', 'Capital Final ($)',
            'Retorno (%)', 'Trades con BE activado'
        ],
        'Valor': [
            len(trades), len(longs), len(shorts),
            f'{win_rate:.1f}', f'{profit_factor:.2f}',
            f'{total_pnl:.2f}', f'{avg_win:.2f}', f'{avg_loss:.2f}',
            f'{max_dd:.2f}', f'{equity[-1]:.2f}',
            f'{(equity[-1]/INITIAL_CAP - 1)*100:.2f}', len(be_trades)
        ]
    })
    display(metrics.style.hide(axis='index'))
else:
    print('No se generaron trades. Revisa los parámetros o los datos.')

## 7. Gráficos

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7.1 — Curva de Equity
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [2, 1, 1]})

# --- Equity ---
ax1 = axes[0]
ax1.plot(df.index[:len(equity)], equity, color='cyan', linewidth=1.2)
ax1.axhline(INITIAL_CAP, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Curva de Equity', fontsize=14, fontweight='bold')
ax1.set_ylabel('Capital ($)')
ax1.fill_between(df.index[:len(equity)], equity, INITIAL_CAP, 
                 where=[e >= INITIAL_CAP for e in equity], color='teal', alpha=0.3)
ax1.fill_between(df.index[:len(equity)], equity, INITIAL_CAP,
                 where=[e < INITIAL_CAP for e in equity], color='red', alpha=0.3)

# --- Distribución de P&L ---
ax2 = axes[1]
if len(trades) > 0:
    colors = ['teal' if t.pnl > 0 else 'red' for t in trades]
    ax2.bar(range(len(trades)), [t.pnl for t in trades], color=colors, alpha=0.8)
    ax2.axhline(0, color='white', linewidth=0.5)
ax2.set_title('P&L por Trade', fontsize=14, fontweight='bold')
ax2.set_ylabel('P&L ($)')

# --- Win rate pie ---
ax3 = axes[2]
if len(trades) > 0:
    be_count = sum(1 for t in trades if abs(t.pnl) < 0.01)
    win_count = len(wins)
    loss_count = len(losses) - be_count
    sizes = [win_count, loss_count, be_count]
    labels = [f'Wins ({win_count})', f'Losses ({loss_count})', f'BE ({be_count})']
    clrs = ['teal', 'red', 'orange']
    # Filter out zeros
    filtered = [(s,l,c) for s,l,c in zip(sizes,labels,clrs) if s > 0]
    if filtered:
        ax3.pie([f[0] for f in filtered], labels=[f[1] for f in filtered], 
                colors=[f[2] for f in filtered], autopct='%1.1f%%', startangle=90)
ax3.set_title('Distribución de Resultados', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7.2 — Gráfico de precio con señales (últimas N velas)
# ═══════════════════════════════════════════════════════════
SHOW_LAST = 500  # cuántas velas mostrar

sub = df.iloc[-SHOW_LAST:].copy()

fig, (ax_price, ax_macd) = plt.subplots(2, 1, figsize=(18, 10), 
                                         gridspec_kw={'height_ratios': [3, 1]},
                                         sharex=True)

# --- Precio y velas ---
up   = sub[sub['close'] >= sub['open']]
down = sub[sub['close'] < sub['open']]

ax_price.vlines(up.index, up['low'], up['high'], color='teal', linewidth=0.5)
ax_price.vlines(down.index, down['low'], down['high'], color='red', linewidth=0.5)
ax_price.bar(up.index, up['close']-up['open'], bottom=up['open'], color='teal', width=0.002, alpha=0.8)
ax_price.bar(down.index, down['open']-down['close'], bottom=down['close'], color='red', width=0.002, alpha=0.8)

# Trendlines
ax_price.plot(sub.index, sub['tl_upper'], color='teal', linestyle='--', alpha=0.6, label='TL Upper')
ax_price.plot(sub.index, sub['tl_lower'], color='red', linestyle='--', alpha=0.6, label='TL Lower')

# ATR SL
ax_price.plot(sub.index, sub['atr_short_sl'], color='red', alpha=0.2, linewidth=1, label='ATR Short SL')
ax_price.plot(sub.index, sub['atr_long_sl'], color='teal', alpha=0.2, linewidth=1, label='ATR Long SL')

# Señales
buys  = sub[sub['buy_signal']]
sells = sub[sub['sell_signal']]
ax_price.scatter(buys.index, buys['low']*0.9998, marker='^', color='lime', s=100, zorder=5, label='BUY')
ax_price.scatter(sells.index, sells['high']*1.0002, marker='v', color='red', s=100, zorder=5, label='SELL')

# FVGs
fvg_sub = sub[sub['fvg_type'] != 0]
for idx, row in fvg_sub.iterrows():
    c = 'teal' if row['fvg_type'] == 1 else 'red'
    ax_price.axhspan(row['fvg_btm'], row['fvg_top'], alpha=0.08, color=c)

ax_price.set_title('Precio + Señales AIO', fontsize=14, fontweight='bold')
ax_price.legend(loc='upper left', fontsize=8)

# --- MACD ---
ax_macd.plot(sub.index, sub['macd'], color='cyan', linewidth=1, label='MACD')
ax_macd.plot(sub.index, sub['macd_sig'], color='orange', linewidth=1, label='Signal')
colors_hist = ['teal' if v > 0 else 'red' for v in sub['macd_hist']]
ax_macd.bar(sub.index, sub['macd_hist'], color=colors_hist, alpha=0.5, width=0.002)
ax_macd.axhline(0, color='white', linewidth=0.3)
ax_macd.set_title('MACD', fontsize=12)
ax_macd.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Tabla de trades

In [ ]:
# ═══════════════════════════════════════════════════════════
# Lista detallada de todos los trades
# ═══════════════════════════════════════════════════════════
if len(trades) > 0:
    trade_data = []
    for t in trades:
        trade_data.append({
            'Dirección': t.direction.upper(),
            'Entrada': f'{t.entry_px:.5f}',
            'SL': f'{t.sl:.5f}',
            'TP': f'{t.tp:.5f}',
            'Salida': f'{t.exit_px:.5f}' if t.exit_px else '-',
            'Lotes': f'{t.lots:.2f}',
            'P&L ($)': f'{t.pnl:.2f}',
            'BE Activo': '✓' if t.be_active else '',
            'Barra Entrada': t.bar_idx,
            'Barra Salida': t.exit_bar if t.exit_bar else '-',
            'Fecha Entrada': str(df.index[t.bar_idx]) if t.bar_idx < len(df) else '-',
        })
    
    trades_df = pd.DataFrame(trade_data)
    
    def color_pnl(val):
        try:
            v = float(val)
            return 'color: lime' if v > 0 else 'color: red' if v < 0 else ''
        except:
            return ''
    
    display(trades_df.style.applymap(color_pnl, subset=['P&L ($)']))
else:
    print('No hay trades para mostrar.')

## 9. Optimización de parámetros (opcional)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Grid search simple sobre RR y ATR Multiplier
# ═══════════════════════════════════════════════════════════
RUN_OPTIMIZATION = False  # Cambiar a True para ejecutar

if RUN_OPTIMIZATION:
    results = []
    rr_range = [2.0, 2.5, 3.0, 3.5, 4.0]
    atr_range = [1.0, 1.5, 2.0, 2.5]
    
    for rr in rr_range:
        for atr_m in atr_range:
            # Recalcular ATR SL
            df_opt = df.copy()
            df_opt['atr_sl_opt'] = df_opt['tr'].ewm(alpha=1/ATR_SL_LEN, adjust=False).mean() * atr_m
            df_opt['atr_short_sl'] = df_opt['high'] + df_opt['atr_sl_opt']
            df_opt['atr_long_sl']  = df_opt['low']  - df_opt['atr_sl_opt']
            
            # Guardar params globales temporalmente
            _rr = RR_RATIO
            RR_RATIO_local = rr
            
            t, eq = run_backtest(df_opt, INITIAL_CAP, RISK_PCT)
            
            pnls = [tr.pnl for tr in t]
            total = sum(pnls) if pnls else 0
            wr = sum(1 for p in pnls if p > 0) / len(pnls) * 100 if pnls else 0
            
            results.append({
                'RR': rr, 'ATR Mult': atr_m,
                'Trades': len(t), 'Win Rate': f'{wr:.1f}%',
                'P&L Total': f'{total:.2f}', 'Capital Final': f'{eq[-1]:.2f}'
            })
    
    opt_df = pd.DataFrame(results)
    display(opt_df.sort_values('P&L Total', ascending=False).head(10))
else:
    print('Optimización desactivada. Cambia RUN_OPTIMIZATION = True para ejecutar.')